# Solutions · 03 · Rate laws and stoichiometry

Worked solutions to the exercises of [notebook 03](../notebooks/03_rate_laws_and_stoichiometry.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engreact is missing): install it from GitHub
    import engreact
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engreact"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engreact import kinetics, thermo
from engreact.kinetics import Reaction
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

Ammonia synthesis N2 + 3H2 <-> 2NH3 with a stoichiometric feed at 450 degC: compute Kp from `thermo`, convert to Kc at 100 bar, and find the equilibrium conversion (gas phase, epsilon = -0.5). Then repeat at 300 bar and at 400 degC to see why the process runs at high pressure.

In [2]:
nh3 = Reaction({"N2": -1, "H2": -3, "NH3": 2}, key="N2")
feed = {"N2": 1.0, "H2": 3.0}
for T_C, p_bar in ((450, 100), (450, 300), (400, 300)):
    T = T_C + 273.15
    Kp = thermo.equilibrium_constant({"N2": -1, "H2": -3, "NH3": 2}, T)          # (bar)^-2
    Kc = thermo.kc_from_kp(Kp, -2, T)                                            # (m3/mol)^2
    C_A0 = 0.25 * p_bar * 1e5 / (8.314 * T)                                      # N2 concentration in the feed, mol/m3
    Xe = kinetics.equilibrium_conversion(nh3, feed, Kc, C_A0, phase="gas")
    print(f"{T_C} degC, {p_bar} bar: Kp = {Kp:.2e} 1/bar2, equilibrium N2 conversion = {Xe:.3f}")

450 degC, 100 bar: Kp = 4.82e-05 1/bar2, equilibrium N2 conversion = 0.275
450 degC, 300 bar: Kp = 4.82e-05 1/bar2, equilibrium N2 conversion = 0.481
400 degC, 300 bar: Kp = 1.78e-04 1/bar2, equilibrium N2 conversion = 0.598


At 450 degC the equilibrium conversion of nitrogen is about 28 % at 100 bar and 48 % at 300 bar (fewer moles
on the right: pressure helps); lowering the temperature to 400 degC raises it to 60 %, but the catalyst slows.
Haber-Bosch runs at 400-500 degC and 150-300 bar with recycle of the unconverted gas - the whole process
is written in these three numbers.

## Exercise 2

For A + B -> C in the liquid phase with C_A0 = 1 mol/L, compare the rate law -r = k C_A C_B as a function of X for Theta_B = 1, 2 and 5. Explain why excess B makes the reaction look first order.

In [3]:
rxn = Reaction({"A": -1, "B": -1, "C": 1}, key="A")
Xs = np.linspace(0, 0.95, 6)
for theta in (1, 2, 5):
    tab = kinetics.stoichiometric_table(rxn, {"A": 1.0, "B": theta}, Xs, "liquid", C_A0=1.0)
    r = tab["C"]["A"] * tab["C"]["B"]
    print(f"Theta_B = {theta}: -r/k = C_A C_B = {np.round(r, 3)}; ratio to first order C_A: {np.round(r / tab['C']['A'], 3)}")

Theta_B = 1: -r/k = C_A C_B = [1.    0.656 0.384 0.185 0.058 0.003]; ratio to first order C_A: [1.   0.81 0.62 0.43 0.24 0.05]
Theta_B = 2: -r/k = C_A C_B = [2.    1.466 1.004 0.615 0.298 0.053]; ratio to first order C_A: [2.   1.81 1.62 1.43 1.24 1.05]
Theta_B = 5: -r/k = C_A C_B = [5.    3.896 2.864 1.905 1.018 0.203]; ratio to first order C_A: [5.   4.81 4.62 4.43 4.24 4.05]


With a large excess of B its concentration barely changes, and $k C_A C_B \approx (k C_{B0}) C_A$: a
pseudo-first-order rate law with an apparent constant $kC_{B0}$. Excess reagent is how second-order kinetics
are measured one reactant at a time - and why the "order" reported depends on the conditions.

## Exercise 3

**Implement it yourself:** write the stoichiometric table for 2A + B -> 3C in the gas phase with a 2:1:1 feed of A:B:inert by hand (Theta, nu, epsilon) and check every column against `kinetics.stoichiometric_table` at X = 0.4.

In [4]:
rxn = Reaction({"A": -2, "B": -1, "C": 3}, key="A")
feed = {"A": 2.0, "B": 1.0, "I": 1.0}
X_ = 0.4
theta_B, theta_C, theta_I = 0.5, 0.0, 0.5
delta = (3 - 2 - 1) / 2                              # 0: no change in moles per mole of A
eps = 0.5 * delta
C_A0 = 1.0
by_hand = {"A": C_A0 * (1 - X_) / (1 + eps * X_), "B": C_A0 * (theta_B - X_ / 2) / (1 + eps * X_), "C": C_A0 * (theta_C + 1.5 * X_) / (1 + eps * X_), "I": C_A0 * theta_I / (1 + eps * X_)}
lib = kinetics.stoichiometric_table(rxn, feed, X_, "gas", C_A0)
print("epsilon:", eps, "library:", lib["epsilon"])
print({s: (round(by_hand[s], 4), round(float(lib["C"][s]), 4)) for s in by_hand})

epsilon: 0.0 library: 0.0
{'A': (0.6, 0.6), 'B': (0.3, 0.3), 'C': (0.6, 0.6), 'I': (0.5, 0.5)}


For 2A + B -> 3C the total moles do not change ($\delta$ = 0), so $\varepsilon$ = 0 and the gas-phase table
reduces to the liquid one; every column matches.